# Протонный диод В. Лившица: модель, протокол опыта, статья 2024 г.
*8 ноября 2026. Источник: `Hydrogen_Thermoel_Gen.pdf` (В. Лившиц, «Новая водородная энергетика», 8 листов).*

Первая код-ячейка `preparser(False)` — для запуска в SageMath 9.6; в обычном Python её можно пропустить.

## 1. Устройство по Лившицу
Одна общая газовая фаза H₂ (без атомизатора). Слои: (1) поглощающий электрод-катализатор H₂ → 2H⁺ + 2e⁻;
(3) протонопроводящая мембрана; (2) выделяющий электрод H⁺ + e⁻ → H, покрытый металлическим барьером,
пропускающим атомы и задерживающим молекулы («АМОН»). Электроны идут по внешней цепи. ЭДС по Нернсту
E = (RT/2F)·ln(P₁/P₂); автор предполагает, что барьер поддерживает «одностороннюю» диффузию при P₁ = P₂.

## 2. Кинетическая модель
Шесть стадий, каждая — закон действующих масс с прямой и обратной скоростью, связанными детальным балансом
своей стадии (это единственное допущение; асимметрия констант разных стадий и сторон — произвольная):

| стадия | реакция | параметры |
|---|---|---|
| 1 | H₂ + 2* ⇄ 2H*(1) | a₁, e₁ |
| 2 | H*(1) ⇄ H⁺(ψ₁) + e⁻(φ=0) | b₁, ε_p |
| 3 | H⁺(ψ₂) + e⁻(φ=V) ⇄ H*(2) | b₂, e₂ |
| 4 | H*(2) ⇄ H(барьер) | k_t, e_b |
| 5 | 2H(барьер) ⇄ H₂ (газ) | k_d |
| 6 | H + O₂(следы) → H₂O, необратимо | k_ox |

Мембрана — омическое сопротивление R_m, внешняя нагрузка R_L. Неизвестные: три заполнения, ψ₁, V.


In [ ]:
preparser(False)  # SageMath 9.6; в чистом Python — закомментировать

In [ ]:
import numpy as np
sig = lambda z: 1/(1+np.exp(-z))

def rates(u, par, P_in, P_out, RL, kox):
    """Стационарные балансы протонного диода. Энергии в RT, потенциалы в RT/F.
    u = (logit θ1, logit θ2, logit θb, ψ1, V)."""
    e1, e2, eb, ep, a1, b1, b2, kt, kd, Rm = par
    l1, l2, lb, psi1, V = u
    t1, t2, tb = sig(l1), sig(l2), sig(lb)
    # 1) H2 + 2* <-> 2H*(1) на поглощающем электроде (детальный баланс: K = exp(-2 e1))
    rads = a1*(P_in*(1-t1)**2 - np.exp(2*e1)*t1**2)
    # 2) H*(1) <-> H+(мембрана, ψ1) + e-(электрод 1, φ=0)
    x1 = e1 - (ep + psi1)
    r1 = b1*(t1*np.exp(x1/2) - (1-t1)*np.exp(-x1/2))
    I = r1                                   # ток во внешней цепи
    psi2 = psi1 - I*Rm                       # омическое падение в мембране
    # 3) H+(ψ2) + e-(электрод 2, φ=V) <-> H*(2) на выделяющем электроде
    x2 = (ep + psi2) - V - e2
    r2 = b2*((1-t2)*np.exp(x2/2) - t2*np.exp(-x2/2))
    # 4) H*(2) <-> H в барьерном слое Pd
    rt = kt*(t2*(1-tb)*np.exp((e2-eb)/2) - tb*(1-t2)*np.exp(-(e2-eb)/2))
    # 5) 2H(барьер) <-> H2 (газ) на внешней поверхности барьера
    rd = kd*(tb**2*np.exp(eb) - P_out*(1-tb)**2*np.exp(-eb))
    # 6) необратимый сток: H + следы O2 -> H2O
    rox = kox*tb
    return np.array([2*rads - r1, r2 - I, rt - r2, rt - 2*rd - rox, V - I*RL]), I

def solve(par, P_in=1.0, P_out=1.0, RL=1e9, kox=0.0, seed=0, starts=8):
    best = None
    for s in range(starts):
        x = np.random.default_rng(seed*100+s).normal(0, 1, 5); x[4] *= 0.01
        for _ in range(300):
            F, _ = rates(x, par, P_in, P_out, RL, kox)
            J = np.zeros((5, 5))
            for i in range(5):
                d = np.zeros(5); d[i] = 1e-7
                J[:, i] = (rates(x+d, par, P_in, P_out, RL, kox)[0] - F)/1e-7
            dx = np.linalg.lstsq(J, -F, rcond=None)[0]; lam = 1.0; f0 = np.linalg.norm(F)
            while lam > 1e-8:
                with np.errstate(all='ignore'):
                    Fn = rates(x+lam*dx, par, P_in, P_out, RL, kox)[0]
                if np.all(np.isfinite(Fn)) and np.linalg.norm(Fn) < f0: break
                lam /= 2
            x = x + lam*dx
            if np.linalg.norm(Fn) < 1e-13: break
        with np.errstate(all='ignore'):
            r = np.linalg.norm(rates(x, par, P_in, P_out, RL, kox)[0])
        if np.isfinite(r) and (best is None or r < best[1]): best = (x, r)
    x, r = best
    return x[4], rates(x, par, P_in, P_out, RL, kox)[1], r

def OK(cond, msg): print(('OK   ' if cond else 'FAIL ') + msg)


### Проверка 1: равные давления, случайная асимметрия

In [ ]:
# Проверка 1: P_in = P_out, без O2, 20 случайных наборов констант (энергии ±3 RT, скорости 1e-2..1e2)
rng = np.random.default_rng(42)
pars = [[*rng.uniform(-3, 3, 4), *(10**rng.uniform(-2, 2, 5)), 10**rng.uniform(-2, 1)] for k in range(20)]
worst_I = 0; worst_res = 0
for k, par in enumerate(pars):
    for RL in [1e-3, 1.0]:                 # короткое замыкание и нагрузка
        V, I, r = solve(par, RL=RL, seed=k+7)
        worst_I = max(worst_I, abs(I)); worst_res = max(worst_res, r)
print('max|I| =', worst_I, ' max невязка =', worst_res)
OK(worst_I < 1e-12, 'ток при равных давлениях равен нулю для всех 20 асимметричных наборов')


### Проверка 2: калибровка законом Нернста

In [ ]:
# Проверка 2 (калибровка): разные давления -> закон Нернста |V| = (1/2) ln(P_out/P_in) в RT/F
par0 = [0.5, -1.0, 1.0, 0.0,  10, 10, 10, 10, 0.01,  0.1]   # сильно асимметричная ячейка
for ratio in [1.01, 1.1, 2.0]:
    V, I, r = solve(par0, P_in=1.0, P_out=ratio, RL=1e9)
    print(f'P_out/P_in={ratio}:  |V|={abs(V):.5f}   (1/2)ln={0.5*np.log(ratio):.5f}')
    OK(abs(abs(V) - 0.5*np.log(ratio)) < 1e-4, f'Нернст при отношении {ratio}')


### Проверка 3: следы кислорода

In [ ]:
# Проверка 3: необратимый сток H + O2 -> H2O на барьере даёт ЭДС и ток (внешний источник свободной энергии)
for kox in [0, 1e-4, 1e-3, 1e-2]:
    Voc, _, _ = solve(par0, RL=1e9, kox=kox); _, Isc, _ = solve(par0, RL=1e-3, kox=kox)
    print(f'kox={kox:g}:  Voc={Voc:+.3e} RT/F = {Voc*25.85:+.3f} мВ (300 K),  Isc={Isc:+.3e}')
OK(abs(solve(par0, RL=1e9, kox=0)[0]) < 1e-9 and solve(par0, RL=1e9, kox=1e-3)[0] > 1e-3,
   'ЭДС появляется только при наличии стока (кислород), без него ноль')


## 3. Результаты (численно)
1. **P₁ = P₂, без O₂**, 20 случайных наборов (энергии ±3 RT, скорости 10⁻²…10², R_m 10⁻²…10):
   |I| < 10⁻¹⁵, |V_oc| < 10⁻⁷ RT/F (численный шум) — **ЭДС и ток равны нулю при любой асимметрии.**
2. **Калибровка**: при P_out/P_in = 1.01; 1.1; 2 получено |V| = ½ln(P_out/P_in) с точностью 10⁻⁵ — закон Нернста.
3. **Сток O₂**: k_ox = 10⁻⁴; 10⁻³; 10⁻² даёт V_oc ≈ 0.09; 0.9; 8 мВ (300 K). То есть любые следы окислителя
   превращают ячейку в топливный элемент — **главный источник ложного положительного результата.**

Причина нуля: сумма сродств всех стадий по замкнутому контуру H₂ → … → H₂ равна
Δμ = RT·ln(P_in/P_out); при P_in = P_out она ноль при любых константах (условие Вегшайдера выполнено
тождественно, поскольку все частицы — один и тот же водород). Барьер меняет скорость, но не ЭДС.


## 4. Протокол решающего опыта
**Ячейка.** Протонный электролит: Nafion 117 (25–80 °C, увлажнение) **или** CsH₂PO₄ (~240 °C) **или**
BaZr₀.₈Y₀.₂O₃ (400–600 °C, без воды — предпочтительно, меньше побочной химии). Поглощающий электрод — Pt
(пористость 30–50 %). Выделяющий электрод — Pt, поверх **сплошная** плёнка Pd или Pd–Ag 1–5 мкм (пор нет;
проверка гелиевым течеискателем). Обе стороны — в **одной** камере H₂ (0.1–1 бар, 6N).

**Чистота.** O₂ < 1 ppm (геттер, проточная очистка); по модели 1 мВ ЭДС ↔ k_ox ~10⁻³ — уровень, который
легко дают следы воздуха. Контроль O₂ датчиком у самой ячейки.

**Изотермичность.** Медный блок, ΔT между электродами < 0.01 K (два термометра Pt100 в контакте с
электродами). Термо-ЭДС водородной концентрационной ячейки — порядка 0.1–1 мВ/K.

**Измерения.** (1) ЭДС разомкнутой цепи электрометром (разрешение ~1 мкВ, вход ≥10¹² Ом), часы–сутки;
(2) ток короткого замыкания; (3) спектр шума и импеданс → T_эфф(f) = S_V/(4k·ReZ).

**Контроли.**
- *Калибровка Нернстом*: разделить камеры, задать P_out/P_in = 1.01 → ожидаемо 0.13 мВ при 300 K.
- *Переворот ячейки*: знак эффекта должен смениться, если он от барьера.
- *Симметричная ячейка* (Pd с обеих сторон или без Pd): эффект от барьера должен исчезнуть.
- *Добавка O₂*: дозированно 1–100 ppm — должна дать ЭДС по модели (проверка чувствительности).
- *Баланс*: интеграл заряда / 2F против расхода H₂ и наработки воды.

**Ожидание.** По модели — ноль. При R ≈ 0.1 Ом·см² (тонкий Nafion) чувствительность по ЭДС ~1 мкВ даёт
ток ~10 мкА/см² — эффект на уровне, заявляемом для ВД2, был бы надёжно виден.
Мощность, *если бы* ЭДС E была: E²/4R ≈ 0.025 / 2.5 / 250 Вт/м² при E = 1 / 10 / 100 мВ.


## 5. Статья 2024 года
Ссылка автора: «Известия Академии электротехнических наук РФ», вып. 1(28) 2024, с. 59–80
(«транзитный осмос», ВД2 с «вентильной средой»). На сайте http://www.acelsc.ru/node/21 есть список выпусков
(№ 28 за 2024 г. присутствует), но полного текста статьи в открытом доступе найти не удалось; в сетевом поиске
статья не находится. **Разобрать её содержание без текста я не могу** — пришлите PDF, и раздел будет дописан.
По описанию в документе (лист 8) это вариант того же класса «устойчиво неравновесных систем» Жвирблиса,
и к нему применим тот же критерий: при равных химических потенциалах на входе и выходе сумма сродств по контуру
равна нулю, ток возможен только при внешнем стоке/источнике.

## 6. Вывод
- Протонный диод — **детально проверяемая** схема: в кинетической модели с произвольной асимметрией стадий
  ЭДС при P₁ = P₂ строго ноль; разность давлений даёт точно Нернста; следы O₂ дают милливольты.
- Опыт однозначен и доступен: изотермическая ячейка с одной камерой H₂, электрометр, контроль O₂.
- Ненулевая устойчивая ЭДС при O₂ < 1 ppm, ΔT < 0.01 K, со сменой знака при перевороте и отсутствием
  в симметричной ячейке была бы настоящим результатом.
